In [ ]:
# ============ step: env ============
import sys, os, subprocess, json
print('python', sys.version.split()[0])
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm==1.0.22', 'kaggle', 'scikit-image'], check=False)
import torch, timm
print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), 'timm', timm.__version__)
print('disk', subprocess.run(['df','-h','/content'],capture_output=True,text=True).stdout.strip().splitlines()[-1])
print('OK_ENV')

In [ ]:
# ============ step: fetch code ============
import subprocess, os
BRANCH = 'main'
REPO = 'https://github.com/akshaytheflash/arch-complementarity.git'
if os.path.exists('/content/repo'):
    subprocess.run(['bash','-lc','cd /content/repo && git fetch -q origin %s && git reset -q --hard origin/%s' % (BRANCH, BRANCH)], check=False)
else:
    subprocess.run(['git','clone','-q','-b',BRANCH,REPO,'/content/repo'], check=True)
print(subprocess.run(['bash','-lc','cd /content/repo && git log --oneline -1 && ls src'],capture_output=True,text=True).stdout)

# VMamba source (official, pinned to main; we record the sha)
if not os.path.exists('/content/vmamba'):
    subprocess.run(['git','clone','-q','--depth','1','https://github.com/MzeroMiko/VMamba.git','/content/vmamba'], check=True)
print(subprocess.run(['bash','-lc','cd /content/vmamba && git log --oneline -1'],capture_output=True,text=True).stdout)
os.environ['VMAMBA_ROOT'] = '/content/vmamba'
sys.path.insert(0, '/content/repo/src')
import models, metrics, datasets, corruption
print('imports OK')

In [ ]:
# ============ step: verify every checkpoint reproduces its published top-1 ============
import os, json, time, torch, numpy as np, sys
from torch.utils.data import DataLoader
sys.path.insert(0,'/content/repo/src')
import models as M

# Kaggle credentials
os.makedirs('/root/.kaggle', exist_ok=True)
if os.path.exists('/content/kaggle.json'):
    open('/root/.kaggle/kaggle.json','w').write(open('/content/kaggle.json').read())
    os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('kaggle creds:', os.path.exists('/root/.kaggle/kaggle.json'))

import datasets as D
t0=time.time()
root = D.kaggle_download('tusonggao/imagenet-validation-dataset')
wnid_root = D.find_wnid_root(root)
print('val root', wnid_root, 'download secs', round(time.time()-t0))

spec = M.REGISTRY['deit-small']
tf, cfg = M.transform_for(spec)
ds = D.WnidFolder(wnid_root, tf, limit=500, seed=0)
print('subset', len(ds), 'cfg', cfg['input_size'], cfg['crop_pct'])
dl = DataLoader(ds, batch_size=64, num_workers=2, pin_memory=True)
mdl = M.load('deit-small')
r = M.infer(mdl, dl)
acc = (r['logits'].argmax(1) == np.zeros(len(r['logits']),int)).mean()
print('deit-small logits', r['logits'].shape, 'feat', r['feat'].shape)
print('OK_DATA')